# Thermal model

The node network in Python, and how it was fitted. No board.

In [1]:
SIMULATED = True          # False, and PORT, at the bench
PORT = 'COM4'

`coaxial.thermal` carries the same network the firmware runs: ten nodes, driver and phase per leg, mcu, regulators, afe, and the board to ambient. Only `board_to_ambient` and `board_capacity` have a clean measurement behind them.

In [2]:
from coaxial import thermal

print('ambient %.1f C' % thermal.AMBIENT)
print('board_to_ambient %.2f K/W, board_capacity %.0f J/K, tau %.1f min'
      % (thermal.CFG['board_to_ambient'], thermal.CFG['board_capacity'],
         thermal.tau_minutes()))
for node in thermal.NODES:
    print('%-12s to_board %5.1f K/W  capacity %.3f J/K'
          % (node, thermal.CFG['to_board'][node], thermal.CFG['capacity'][node]))

ambient 20.0 C
board_to_ambient 8.33 K/W, board_capacity 49 J/K, tau 6.8 min
driver_u     to_board  45.6 K/W  capacity 0.117 J/K
driver_v     to_board  45.6 K/W  capacity 0.117 J/K
driver_w     to_board  45.6 K/W  capacity 0.117 J/K
phase_u      to_board  45.6 K/W  capacity 0.400 J/K
phase_v      to_board  45.6 K/W  capacity 0.400 J/K
phase_w      to_board  45.6 K/W  capacity 0.400 J/K
mcu          to_board  22.5 K/W  capacity 0.900 J/K
regulators   to_board  15.0 K/W  capacity 0.800 J/K
afe          to_board  41.5 K/W  capacity 0.300 J/K


The NTC sits in the drivers' hot spot: an offset over the board taken in the passive state, and a coupling to the drivers' rise solved from the switching state.

In [3]:
print(thermal.MEASURED)
print('NTC_OFFSET %.2f K' % thermal.NTC_OFFSET)
print('NTC_SEES_DRIVERS %.3f' % thermal.NTC_SEES_DRIVERS)
print('driver rise while switching %.1f K' % thermal.DRIVER_RISE_SWITCHING)
for state in thermal.STATES:
    print('%-8s %s' % (state, thermal.STATE_IS[state]))

{'passive': {'ntc': 36.0, 'board': 30.0}, 'switching': {'ntc': 55.6, 'board': 40.0}}
NTC_OFFSET 6.00 K
NTC_SEES_DRIVERS 1.055
driver rise while switching 9.1 K
passive  AFE off: the drivers have supply, no PWM
afe      AFE on: drivers unpowered, sensors alive, no traffic
traffic  AFE on: DAQ at full tilt, data off the board
switch   AFE off: three legs at 50 %


In [4]:
steady = thermal.steady(thermal.POWER_SWITCHING)
print('power while switching: %.2f W' % sum(thermal.POWER_SWITCHING.values()))
for node in thermal.ALL_NODES:
    print('%-12s %6.2f C' % (node, steady[node]))
print('NTC expected %.2f C' % thermal.expected_ntc(steady['board'], steady['driver_v'] - steady['board']))
for minutes in (5, 10, 25):
    print('%2d min: %.0f %% of the way to equilibrium' % (minutes, 100 * thermal.settled_fraction(minutes)))

power while switching: 2.40 W
driver_u      49.11 C
driver_v      49.11 C
driver_w      49.11 C
phase_u       39.99 C
phase_v       39.99 C
phase_w       39.99 C
mcu           54.98 C
regulators    57.00 C
afe           39.99 C
board         39.99 C
NTC expected 55.61 C
 5 min: 52 % of the way to equilibrium
10 min: 77 % of the way to equilibrium
25 min: 97 % of the way to equilibrium


`calibrate` is the fit itself: `to_board = (T_zone - T_reference) / P_zone`, one division per node, the camera's surface temperature at each source against a reference patch of soldermask at the same moment. Feed it a zone and the power in that zone and it returns that zone's spreading resistance.

In [5]:
camera = {'mcu': 40.0 + 20.0, 'regulators': 40.0 + 10.1}
for node, k_per_w in sorted(thermal.calibrate(camera, board_c=40.0).items()):
    print('%-12s %6.1f K/W from the camera, %5.1f in the model'
          % (node, k_per_w, thermal.CFG['to_board'][node]))

mcu            30.0 K/W from the camera,  22.5 in the model
regulators      8.9 K/W from the camera,  15.0 in the model


In [6]:
from coaxial import thermalmap

print(thermalmap.render({n: steady[n] for n in thermal.NODES}, steady['board'],
                        cells=60, colour=False, title='steady state, switching'))

  steady state, switching

                                                cccccccccccccccccccccccc                                                  @@ 100 C
                                        cccccccccccccccccccccccccccccccccccccccc                                          WW
                                  cccccccccccccccccccccccccccccccccccccccccccccccccccc                                    WW
                              cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc                                88
                          cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc                            88
                        cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc                          88
                    cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc                      %%
                  cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc

## Conclusions

In [7]:
print('measured, against the supply and the camera:')
for name in ('board_to_ambient', 'board_capacity'):
    print('   %-20s %.2f' % (name, thermal.CFG[name]))
print()
print('the drivers, and the chain the NTC compensation hangs on:')
watts = sum(thermal.POWER_SWITCHING[n] for n in thermal.DRIVERS)
lumped = thermal.DRIVER_RISE_SWITCHING / watts
print('   %-20s %.2f W over the three legs' % ('driver power', watts))
print('   %-20s %.1f K/W lumped, %.1f per leg'
      % ('to_board', lumped, thermal.CFG['to_board']['driver_u']))
print('   %-20s %.2f W x %.1f K/W = %.1f K'
      % ('node rise switching', watts, lumped, thermal.DRIVER_RISE_SWITCHING))
print('   %-20s %.1f - %.1f - %.1f = %.1f K over the node'
      % ('NTC while switching', thermal.MEASURED['switching']['ntc'],
         thermal.MEASURED['switching']['board'], thermal.NTC_OFFSET,
         thermal.MEASURED['switching']['ntc'] - thermal.MEASURED['switching']['board']
         - thermal.NTC_OFFSET))
print('   %-20s %.3f of that rise'
      % ('NTC_SEES_DRIVERS', thermal.NTC_SEES_DRIVERS))

measured, against the supply and the camera:
   board_to_ambient     8.33
   board_capacity       49.00

the drivers, and the chain the NTC compensation hangs on:
   driver power         0.60 W over the three legs
   to_board             15.2 K/W lumped, 45.6 per leg
   node rise switching  0.60 W x 15.2 K/W = 9.1 K
   NTC while switching  55.6 - 40.0 - 6.0 = 9.6 K over the node
   NTC_SEES_DRIVERS     1.055 of that rise


No least squares: `to_board = (T_zone - T_reference) / P_zone`, one division per node, T from a camera against a dead patch of soldermask - not the NTC, which sits in the drivers' hot spot. A spreading resistance in the laminate is a few K/W; tens means the power or the reference surface is wrong.

Each state adds one power term to the one before, so the differences isolate a subsystem no single state can. Each was held 25 minutes, 3.7 times the board's constant.

**The camera saw one bridge zone**, so it constrains the three legs together. Per leg is three times the lumped 15.2 K/W and a third of the capacity: the three in parallel are what was measured, while one leg alone rises three times as far and three times as fast.

The NTC coupling is above 1 because it sits closer to the heat than the point its node stands for; capping it at 1.0 cost 5.6 K in the switching state. The whole calibration was taken **dry**, and at 100 A the shunt alone makes 35 W against the dry budget's 1.2 W.

## The envelope

Four states the board actually sits in, from the same network: quiet, switching dry, switching under current, and cooling. The housekeeping - MCU and regulators - is there in all of them; the drivers' share appears with the PWM, and conduction with the current.

In [8]:
import math
from coaxial import inverter

R_PHASE = inverter.RDS_ON + inverter.SHUNT
KT = 0.0435                          # N.m/A, coaxial.motor.KT_NM_PER_AMP

def split(iq=0.0, switching=True):
    """Power per node: housekeeping, the drivers' share, conduction."""
    out = dict(thermal.POWER_SWITCHING)
    if not switching:
        for n in thermal.DRIVERS:
            out[n] = 0.0
    rms = iq / math.sqrt(2.0)
    for n in thermal.PHASES:
        out[n] = rms * rms * R_PHASE
    return out

CEILING_C = 125.0 * 0.85             # the record's throttle point
STATES = (('quiet, no PWM', 0.0, False),
          ('switching, no current', 0.0, True),
          ('switching, 10 A of iq', 10.0, True),
          ('switching, 20 A of iq', 20.0, True),
          ('switching, 60 A of iq', 60.0, True))

print('%-24s %6s %8s %10s %11s   %s'
      % ('', 'W', 'board C', 'worst C', 'which', 'holdable'))
for name, iq, on in STATES:
    power = split(iq, on)
    at = thermal.steady(power)
    worst = max(thermal.NODES, key=lambda n: at[n])
    holds = at[worst] <= CEILING_C
    print('%-24s %6.2f %8.1f %10.1f %11s   %s'
          % (name, sum(power.values()), at['board'], at[worst],
             thermal.pretty(worst),
             'yes' if holds else 'NO - a burst, timed below'))

                              W  board C    worst C       which   holdable
quiet, no PWM              1.80     35.0       52.0  regulators   yes
switching, no current      2.40     40.0       57.0  regulators   yes
switching, 10 A of iq      3.19     46.6       63.6  regulators   yes
switching, 20 A of iq      5.58     66.5      114.8     phase U   NO - a burst, timed below
switching, 60 A of iq     31.02    278.4      713.4     phase U   NO - a burst, timed below


Where the worst node's equilibrium reaches the throttle point is the current the board can hold for ever; everything above it is timed.

In [9]:
rms = thermal.continuous_amps(R_PHASE, CEILING_C)
iq_cont = rms * math.sqrt(2.0)
at = thermal.steady(thermal.phase_power(rms, R_PHASE))
print('continuous: %.1f A rms a phase = %.1f A of iq = %.2f N.m'
      % (rms, iq_cont, KT * iq_cont))
print('   worst node %.1f C against the %.1f C throttle point, board %.1f C'
      % (max(at[n] for n in thermal.NODES), CEILING_C, at['board']))

continuous: 15.1 A rms a phase = 21.3 A of iq = 0.93 N.m
   worst node 106.2 C against the 106.2 C throttle point, board 70.0 C


Equilibrium is where a holdable state ends up. The board's own constant decides how long that takes, and a node reaches its rise over the board far sooner:

In [10]:
tau_board = thermal.tau_minutes()
print('board       %.0f J/K over %.2f K/W = %.1f min'
      % (thermal.CFG['board_capacity'], thermal.CFG['board_to_ambient'], tau_board))
for node in ('driver_u', 'phase_u', 'mcu'):
    tau = thermal.CFG['capacity'][node] * thermal.CFG['to_board'][node]
    print('%-11s %.2f J/K over %.1f K/W = %.1f s'
          % (thermal.pretty(node), thermal.CFG['capacity'][node],
             thermal.CFG['to_board'][node], tau))
print()
for minutes in (1, 5, 10, 25):
    print('%2d min: %.0f %% of the way to equilibrium'
          % (minutes, 100 * thermal.settled_fraction(minutes)))

board       49 J/K over 8.33 K/W = 6.8 min
driver U    0.12 J/K over 45.6 K/W = 5.3 s
phase U     0.40 J/K over 45.6 K/W = 18.2 s
mcu         0.90 J/K over 22.5 K/W = 20.2 s

 1 min: 14 % of the way to equilibrium
 5 min: 52 % of the way to equilibrium
10 min: 77 % of the way to equilibrium
25 min: 97 % of the way to equilibrium


A burst is over long before either constant matters: the phase node climbs at `P / capacity` from wherever it started, and what stops it is the ceiling in the calibration record, throttled at 85 %.

In [11]:
capacity = thermal.CFG['capacity']['phase_u']

print('  iq A   W a phase   K/s      s from ambient   s from a warm board (60 C)')
for iq in (20.0, 40.0, 60.0, 100.0):
    p = (iq / math.sqrt(2.0)) ** 2 * R_PHASE
    slope = p / capacity
    print('%7.0f %11.1f %7.1f %16.2f %27.2f'
          % (iq, p, slope, (CEILING_C - thermal.AMBIENT) / slope,
             (CEILING_C - 60.0) / slope))

  iq A   W a phase   K/s      s from ambient   s from a warm board (60 C)
     20         1.1     2.6            32.55                       17.45
     40         4.2    10.6             8.14                        4.36
     60         9.5    23.8             3.62                        1.94
    100        26.5    66.2             1.30                        0.70


And cooling: with the current off the node dumps into the board on its own constant, and the board loses what it has to ambient on 6.8 minutes. The node is cold in seconds; the board is what a second burst has to wait for.

In [12]:
def cools_in(rise_k, tau_s, to_k=1.0):
    """Seconds for an exponential fall from rise_k down to to_k."""
    return tau_s * math.log(rise_k / to_k) if rise_k > to_k else 0.0

tau_node = capacity * thermal.CFG['to_board']['phase_u']
print('phase node, after a 60 A burst to the ceiling:')
rise = CEILING_C - thermal.AMBIENT
for target in (20.0, 5.0, 1.0):
    print('   to +%4.0f K over the board: %5.1f s' % (target, cools_in(rise, tau_node, target)))
print()
print('board, after holding a state it can hold:')
for name, iq, on in STATES:
    at = thermal.steady(split(iq, on))
    if max(at[n] for n in thermal.NODES) > CEILING_C:
        continue
    board_rise = at['board'] - thermal.AMBIENT
    print('   %-24s +%5.1f K, back to +1 K in %.0f min'
          % (name, board_rise, cools_in(board_rise, tau_board * 60.0) / 60.0))

phase node, after a 60 A burst to the ceiling:
   to +  20 K over the board:  26.7 s
   to +   5 K over the board:  51.9 s
   to +   1 K over the board:  81.3 s

board, after holding a state it can hold:
   quiet, no PWM            + 15.0 K, back to +1 K in 18 min
   switching, no current    + 20.0 K, back to +1 K in 20 min
   switching, 10 A of iq    + 26.6 K, back to +1 K in 22 min


The two constants are what the whole envelope rests on. A burst lives on the node's 18 seconds and is bounded by the record's ceiling; the duty cycle a bench can hold lives on the board's 6.8 minutes, which is also what a second burst waits for. Between them the observer runs at 100 ms steps and samples the NTC every 30 s, which is fast against the board and slow against a burst - so the board's estimate is anchored and the node's is open-loop over the burst, by construction.

The continuous number falls out of the first table: at the current where the worst node's equilibrium reaches the throttle point, the state can be held for ever, and everything above it is timed.